### pocet center, size 계산 ###

In [2]:
from pathlib import Path
import csv

# ===== 사용자 설정 =====
MODE = "A"  # "A" = 리간드 파일 1개로 계산, "B" = 단백질 PDB에서 리간드 resname 추출
OUT_CSV = Path("/home/jeongin/eupatilin/data/KMU-11421/pocket_info/8TB5_pocket_info.csv")  # 결과 CSV 경로

# (A) 모드: 리간드 파일 경로 (PDB 또는 PDBQT)
LIGAND_FILE = Path("/home/jeongin/eupatilin/data/KMU-11421/split_pdb/8TB5_ligand.pdb")  

# (B) 모드: 단백질 PDB + 리간드 resname (3-letter)
PROTEIN_PDB = Path("/home/jeongin/eupatilin/data/KMU-11421/split_pdb/8TB5_protein.pdb")

# 박스 설정
PAD = 5.0       # 각 축으로 ±PAD Å 패딩 (size = (max-min) + 2*PAD)
MIN_SIZE = 20.0 # 각 축 최소 박스 길이(Å). 20Å 기본
MAKE_CUBE = False  # True면 size_x=y=z=세 축 중 최댓값으로 큐브화(옵션)


def read_coords_from_pdb_like(path: str):
    """PDB/PDBQT에서 ATOM/HETATM 좌표 읽기"""
    xs, ys, zs = [], [], []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if not (line.startswith("ATOM") or line.startswith("HETATM")):
                continue
            try:
                x = float(line[30:38]); y = float(line[38:46]); z = float(line[46:54])
            except ValueError:
                continue
            xs.append(x); ys.append(y); zs.append(z)
    return xs, ys, zs

def bbox_center_size(xs, ys, zs, pad=5.0, min_size=10.0, make_cube=False):
    """좌표의 AABB로 center/size 계산"""
    if not xs:
        raise ValueError("좌표가 비었습니다. 입력을 확인하세요.")
    minx, maxx = min(xs), max(xs)
    miny, maxy = min(ys), max(ys)
    minz, maxz = min(zs), max(zs)
    cx, cy, cz = (minx+maxx)/2.0, (miny+maxy)/2.0, (minz+maxz)/2.0
    sx = max((maxx-minx) + 2.0*pad, min_size)
    sy = max((maxy-miny) + 2.0*pad, min_size)
    sz = max((maxz-minz) + 2.0*pad, min_size)
    if make_cube:
        m = max(sx, sy, sz)
        sx = sy = sz = m
    return (cx, cy, cz), (sx, sy, sz)


# === 실행 ===
assert LIGAND_FILE.exists(), f"리간드 파일 없음: {LIGAND_FILE}"
xs, ys, zs = read_coords_from_pdb_like(str(LIGAND_FILE))
center, size = bbox_center_size(xs, ys, zs, pad=PAD, min_size=MIN_SIZE, make_cube=MAKE_CUBE)

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with OUT_CSV.open("w", newline="") as w:
    writer = csv.DictWriter(w, fieldnames=["center_x","center_y","center_z","size_x","size_y","size_z","pad","min_size","cube"])
    writer.writeheader()
    writer.writerow({
        "center_x": center[0], "center_y": center[1], "center_z": center[2],
        "size_x": size[0], "size_y": size[1], "size_z": size[2],
        "pad": PAD, "min_size": MIN_SIZE, "cube": int(MAKE_CUBE)
    })

print("✅ pocket_info.csv 저장 완료")
print(f"center=({center[0]:.3f},{center[1]:.3f},{center[2]:.3f})  size=({size[0]:.3f},{size[1]:.3f},{size[2]:.3f})")

✅ pocket_info.csv 저장 완료
center=(36.875,-11.093,27.758)  size=(43.410,31.555,63.708)


In [12]:
import pandas as pd
import re
import os
import sys
from pathlib import Path

OUT_DIR = Path("/home/jeongin/eupatilin/data/negative_protein/pocket_info/GR")
POCKET_DIR = Path("/home/jeongin/eupatilin/data/negative_protein/pocket_info/GR")

def build_pocket_master(pocket_dir: Path, out_csv: Path):
    rows = []
    for csv in pocket_dir.glob("*pock*et_info.csv"):
        m = re.match(r"([0-9A-Za-z]+)_pock.*et_info\.csv$", csv.name)
        pdb_id = m.group(1) if m else None
        if not pdb_id: 
            continue
        df = pd.read_csv(csv)
        if not set(["center_x","center_y","center_z","size_x","size_y","size_z"]).issubset(df.columns):
            print(f"[WARN] 컬럼 부족: {csv.name}")
            continue
        r = df.iloc[0]  # 구조당 1행 가정
        rows.append({
            "pdb_id": pdb_id,
            "center_x": r["center_x"], "center_y": r["center_y"], "center_z": r["center_z"],
            "size_x": r["size_x"], "size_y": r["size_y"], "size_z": r["size_z"]
        })
    master = pd.DataFrame(rows).drop_duplicates("pdb_id")
    master.to_csv(out_csv, index=False)
    return master

POCKET_MASTER = OUT_DIR / "GR_pocket_info.csv"
master = build_pocket_master(POCKET_DIR, POCKET_MASTER)
print(f"[OK] pocket_master 생성: {POCKET_MASTER} (행 {len(master)})")


[OK] pocket_master 생성: /home/jeongin/eupatilin/data/negative_protein/pocket_info/GR/GR_pocket_info.csv (행 6)
